# 08 · Decision trees & random forests

A **decision tree** grows by repeatedly asking the single best yes/no question
about the data: for every candidate split — a threshold on a numeric feature,
or a category on a categorical one — it measures how much purer the two
resulting groups are and greedily keeps the split with the largest impurity
reduction. It recurses until a stopping rule kicks in (`max_depth`,
`min_samples_to_split`, or a pure node); each leaf then predicts from the
training examples that landed there. The same machinery drives both tasks —
only `impurity_measure` changes:

- **Classification** — `"gini"` or `"entropy"` score how mixed the classes in
  a node are; a leaf predicts the **majority class**.
- **Regression** — `"mse"` scores how much a split reduces the **variance** of
  the target; a leaf predicts the **mean** target of the rows that reached it.

A single tree is a **greedy, high-variance** model: small changes in the
training data can flip which split looks best near the root and cascade into a
very different tree. A **random forest** tames that variance by growing many
trees on **bootstrapped** samples of the rows, each restricted to a **random
subset of the columns** (`n_feat`), and combining their predictions — majority
vote for classification, averaging for regression. Individual trees stay
noisy; decorrelated together, they aren't.

## Data and a train/test split

As with [`06_knn`](06_knn.ipynb), we fit on a training split and evaluate on a
held-out test split. Two features (`x1`, `x2`) actually drive the label; four
more (`noise1`–`noise4`) are pure noise, so overfitting has somewhere to
hide.

In [1]:
import numpy as np
import polars as pl

from dolcestat.preprocessing import DolceSet
from dolcestat.trees import DecisionTree, RandomForest

rng = np.random.default_rng(0)
n = 400
x1 = rng.normal(0, 1, n)
x2 = rng.normal(0, 1, n)
noise = rng.normal(0, 1, (n, 4))
prob = 1 / (1 + np.exp(-(1.5 * x1 - 2.0 * x2 + 0.4)))
label = (rng.uniform(size=n) < prob).astype(int)

cols = {"x1": x1, "x2": x2}
cols.update({f"noise{i + 1}": noise[:, i] for i in range(4)})
cols["label"] = label
df = pl.DataFrame(cols)

order = rng.permutation(n)
train_rows, test_rows = order[:300].tolist(), order[300:].tolist()
train = DolceSet()
train.load_from_polars_dataframe(df[train_rows], target_col="label")
test = DolceSet()
test.load_from_polars_dataframe(df[test_rows], target_col="label")

## Fitting a single tree

Build a `DecisionTree` from the training set, `fit()`, then `predict` on the
test set. Like every estimator here, `predict` hands back an **analyzer**
([`07_metrics`](07_metrics.ipynb)) — `y_fit` holds the predicted class and
`accuracy`/`precision`/etc. read straight off it.

In [2]:
tree = DecisionTree(train, max_depth=4, min_samples_to_split=5)
tree.fit()
result = tree.predict(test)

print("predicted labels:", result.y_fit[:10])
print(f"test accuracy: {result.accuracy(0.5):.3f}")

predicted labels: [1 0 1 1 0 1 1 1 0 0]
test accuracy: 0.800


## The knobs

- **`max_depth`** — how many splits deep the tree can grow. Shallow trees
  underfit; unlimited depth keeps splitting until every leaf is pure (or,
  for regression, until each leaf is tiny), memorising the training set
  (including its noise features).
- **`min_samples_to_split`** — a node with fewer samples than this becomes a
  leaf instead of splitting further, another brake on overfitting.
- **`impurity_measure`** — `"gini"` (default) or `"entropy"` for
  classification, `"mse"` for regression (see the regression section below).
- **`categorical_features`** — column indices to split by *equality*
  (`feature == category`) instead of a numeric `<=` threshold.

In [3]:
for depth in [1, 2, 4, 8, None]:
    t = DecisionTree(train, max_depth=depth, min_samples_to_split=5)
    t.fit()
    train_acc = t.predict(train).accuracy(0.5)
    test_acc = t.predict(test).accuracy(0.5)
    print(f"max_depth={depth!s:>4}  train_acc={train_acc:.3f}  test_acc={test_acc:.3f}")

max_depth=   1  train_acc=0.767  test_acc=0.680


max_depth=   2  train_acc=0.777  test_acc=0.720


max_depth=   4  train_acc=0.853  test_acc=0.800


max_depth=   8  train_acc=0.963  test_acc=0.660


max_depth=None  train_acc=0.970  test_acc=0.660


### Categorical splits

Pass column indices via `categorical_features` to split those columns by
category membership rather than a threshold — useful when a feature's numeric
encoding (e.g. a label-encoded category `0`, `1`, `2`) carries no real
order.

In [4]:
cat = rng.integers(0, 3, n).astype(float)  # 3 categories, label-encoded
cat_effect = np.where(cat == 0, 2.0, np.where(cat == 1, -1.0, 0.5))
cat_noise = rng.normal(0, 1, n)
cat_prob = 1 / (1 + np.exp(-(cat_effect + 0.3 * cat_noise)))
cat_label = (rng.uniform(size=n) < cat_prob).astype(int)
cat_df = pl.DataFrame({"category": cat, "noise": cat_noise, "label": cat_label})
cat_data = DolceSet()
cat_data.load_from_polars_dataframe(cat_df, target_col="label")

cat_tree = DecisionTree(cat_data, max_depth=3, categorical_features=[0])
cat_tree.fit()
print("train accuracy:", round(cat_tree.predict(cat_data).accuracy(0.5), 3))
print("root split is categorical:", cat_tree.tree.is_categorical_split)

train accuracy: 0.772
root split is categorical: True


## Random forests: bagging + feature subsampling

`RandomForest` grows `n_trees` trees, each on a **bootstrap sample** of the
rows and a **random subset of `n_feat` columns**. Restricting each tree to
different features decorrelates their mistakes, so majority voting across the
forest cancels out more noise than any single tree's errors would suggest.
`predict` returns the same kind of analyzer as a single tree.

In [5]:
deep_tree = DecisionTree(train, max_depth=None, min_samples_to_split=2)
deep_tree.fit()
print("single deep tree   train_acc=%.3f  test_acc=%.3f" % (
    deep_tree.predict(train).accuracy(0.5), deep_tree.predict(test).accuracy(0.5)))

rf = RandomForest(train, n_trees=100, n_feat=3, max_depth=None, min_samples_to_split=2)
rf.fit()
print("random forest       train_acc=%.3f  test_acc=%.3f" % (
    rf.predict(train).accuracy(0.5), rf.predict(test).accuracy(0.5)))

single deep tree   train_acc=1.000  test_acc=0.720


random forest       train_acc=1.000  test_acc=0.760


### More trees, less variance

Each individual tree in the forest is still left free to overfit
(`max_depth=None`); what controls the *forest's* generalisation is `n_trees` —
more trees average away more of that per-tree noise, with diminishing
returns.

In [6]:
for n_trees in [1, 5, 20, 100]:
    forest = RandomForest(
        train, n_trees=n_trees, n_feat=3, max_depth=None, min_samples_to_split=2
    )
    forest.fit()
    test_acc = forest.predict(test).accuracy(0.5)
    print(f"n_trees={n_trees:>4}  test_acc={test_acc:.3f}")

n_trees=   1  test_acc=0.540


n_trees=   5  test_acc=0.710


n_trees=  20  test_acc=0.760


n_trees= 100  test_acc=0.810


## Regression trees

Set `impurity_measure="mse"` and the exact same `DecisionTree`/`RandomForest`
classes grow **regression** trees instead: each candidate split is scored by
how much it reduces the target's variance, and each leaf predicts the mean
target of the training rows that reached it. `predict` now hands back a
`RegressionAnalyzer` ([`07_metrics`](07_metrics.ipynb)) — read off `rmse()`,
`mape()`, and friends instead of `accuracy()`.

New data: apartment price from size and room count (as in
[`07_metrics`](07_metrics.ipynb)), plus three noise features so overfitting has
somewhere to hide here too.

In [7]:
reg_rng = np.random.default_rng(2)
m = 400
size_m2 = reg_rng.uniform(45, 190, m)
rooms = reg_rng.integers(1, 6, m).astype(float)
reg_noise = reg_rng.normal(0, 1, (m, 3))
price_k = 3.0 * size_m2 + 25 * rooms + 60 + reg_rng.normal(0, 18, m)

reg_cols = {"size_m2": size_m2, "rooms": rooms}
reg_cols.update({f"noise{i + 1}": reg_noise[:, i] for i in range(3)})
reg_cols["price_k"] = price_k
reg_df = pl.DataFrame(reg_cols)

reg_order = reg_rng.permutation(m)
reg_train = DolceSet()
reg_train.load_from_polars_dataframe(reg_df[reg_order[:300].tolist()], target_col="price_k")
reg_test = DolceSet()
reg_test.load_from_polars_dataframe(reg_df[reg_order[300:].tolist()], target_col="price_k")

reg_tree = DecisionTree(reg_train, max_depth=4, min_samples_to_split=5, impurity_measure="mse")
reg_tree.fit()
reg_result = reg_tree.predict(reg_test)

print("predicted prices:", np.round(reg_result.y_fit[:6], 1))
print("true prices:     ", np.round(reg_result.y_true[:6], 1))
print(f"test RMSE: {reg_result.rmse():.2f} (thousand €)")
print(f"test MAPE: {reg_result.mape():.2f}%")

predicted prices: [596.4 501.3 668.9 369.7 385.4 515.7]
true prices:      [558.3 494.3 646.  399.4 429.4 505.5]
test RMSE: 32.42 (thousand €)
test MAPE: 5.41%


### Random forest regression — `n_feat` matters more here

Bagging still trades variance for bias, but watch `n_feat`: this dataset's
price is driven almost entirely by two features (`size_m2`, `rooms`) out of
five. Sample too few columns per tree and some trees never see either
informative feature — that *hurts* the ensemble instead of helping it. Feature
subsampling only pays off once `n_feat` leaves most trees a real shot at the
signal.

In [8]:
deep_reg_tree = DecisionTree(
    reg_train, max_depth=None, min_samples_to_split=2, impurity_measure="mse"
)
deep_reg_tree.fit()
print("single deep tree   test_rmse=%.2f" % deep_reg_tree.predict(reg_test).rmse())

for n_feat in [2, 3, 4, 5]:
    reg_forest = RandomForest(
        reg_train,
        n_trees=100,
        n_feat=n_feat,
        max_depth=None,
        min_samples_to_split=2,
        impurity_measure="mse",
    )
    reg_forest.fit()
    test_rmse = reg_forest.predict(reg_test).rmse()
    print(f"n_feat={n_feat}  random forest  test_rmse={test_rmse:.2f}")

single deep tree   test_rmse=29.83


n_feat=2  random forest  test_rmse=82.01


n_feat=3  random forest  test_rmse=51.45


n_feat=4  random forest  test_rmse=33.49


n_feat=5  random forest  test_rmse=19.41
